# Qwen-27B Mediator-Matched Handoff Symmetry

The previous split-sample experiment found that an L48 four-component PLS state reliably moved L49, L55, and the answer in the physically correct direction, but with only about one quarter of the downstream effect produced by a direct L49 intervention.

[实验设置与运行说明](../../../docs/experiment_notes/11_handoff_development_and_matching/00a69334c94bf7d9.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')


## 1. Fixed route, matched gain, and mirror pairs


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import json
import os
import signal
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)

ANALYSIS_STACK = {
    'numpy': '2.5.1',
    'scipy': '1.18.0',
    'pandas': '3.0.5',
    'matplotlib': '3.11.1',
    'scikit-learn': '1.9.0',
    'tqdm': '4.67.1',
}
ENV_MARKER = Path('/content/visualtorque_inference_analysis_stack_v3.json')
probe_code = '''
import json
import matplotlib
import numpy
import pandas
import scipy
import sklearn
import tqdm
print(json.dumps({
    'numpy': numpy.__version__,
    'scipy': scipy.__version__,
    'pandas': pandas.__version__,
    'matplotlib': matplotlib.__version__,
    'scikit-learn': sklearn.__version__,
    'tqdm': tqdm.__version__,
}, sort_keys=True))
'''

def fresh_process_probe():
    return subprocess.run(
        [sys.executable, '-c', probe_code],
        text=True, capture_output=True, check=False,
    )

try:
    marker = json.loads(ENV_MARKER.read_text(encoding='utf-8'))
except (FileNotFoundError, json.JSONDecodeError):
    marker = None
probe = fresh_process_probe()
stack_ready = (
    marker == ANALYSIS_STACK
    and probe.returncode == 0
    and json.loads(probe.stdout.strip().splitlines()[-1]) == ANALYSIS_STACK
)
if not stack_ready:
    cell_stage('Repairing the pinned binary analysis stack')
    packages = [
        f'{name}=={version}' for name, version in ANALYSIS_STACK.items()
    ]
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--no-cache-dir', '--force-reinstall', '--only-binary=:all:',
        *packages,
    ])
    repaired = fresh_process_probe()
    if repaired.returncode != 0:
        raise RuntimeError(
            'The repaired stack still fails in a fresh Python process.\n'
            + repaired.stderr[-4000:]
        )
    repaired_versions = json.loads(
        repaired.stdout.strip().splitlines()[-1]
    )
    assert repaired_versions == ANALYSIS_STACK, repaired_versions
    ENV_MARKER.write_text(
        json.dumps(ANALYSIS_STACK, sort_keys=True), encoding='utf-8'
    )
    print(
        '\nPinned binary packages were repaired. Colab will restart once. '
        'After it reconnects, choose Runtime > Run all. This is expected.',
        flush=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)

cell_stage('Installing pinned inference dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34',
])
verified = fresh_process_probe()
assert verified.returncode == 0, verified.stderr[-4000:]
assert json.loads(verified.stdout.strip().splitlines()[-1]) == ANALYSIS_STACK

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('numpy / scipy / sklearn:',
      np.__version__, scipy.__version__, sklearn.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify GPU and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock mediator matching and mirror symmetry
cell_stage('Freezing parent route, global gain, pairs, conditions, and equivalence margins')
import hashlib
import json
from datetime import datetime, timezone

MODEL_KEY = 'qwen27b'
MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_SLUG = 'Qwen__Qwen3.5-27B__373612cb'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
UPSTREAM_LAYER = 48
UPSTREAM_METHOD = 'pls4_prototype'
SEMANTIC_LAYER = 49
LABEL_LAYER = 55
CAPTURE_LAYERS = (UPSTREAM_LAYER, SEMANTIC_LAYER, LABEL_LAYER)
SITE_SCOPE = 'final_prompt_token'

SEED = 20261107
DIRECTION_SEED = 20260901
LABELS = ('A', 'B', 'C')
MAPPING_VARIANTS = ('forward', 'reverse')
DONOR_TASKS = ('weight_more', 'weight_less')
QUESTION_KINDS = (
    'outcome', 'left_units', 'right_units', 'left_distance',
    'right_distance', 'left_torque', 'right_torque', 'net_torque',
)
BOOTSTRAP_REPLICATES = 10000
ALPHA_LEVEL = 0.05
EQUIVALENCE_FRACTION = 0.25
MEDIATOR_MATCH_FRACTION = 0.20
PATCH_CONDITION_BATCH_SIZE = 1
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 0.02

PARENT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_nettorque_semantic_handoff_mediation_v1'
)
PARENT_RUN_PATH = PARENT_ROOT / 'RUN_COMPLETE.json'
PARENT_PROTOCOL_PATH = PARENT_ROOT / 'frozen_protocol.json'
PARENT_SUPPORT_PATH = PARENT_ROOT / 'frozen_support.jsonl'
PARENT_SPLIT_PATH = PARENT_ROOT / 'discovery_confirmation_split.json'
PARENT_ROUTE_PATH = PARENT_ROOT / 'selected_upstream_route.json'
PARENT_DIRECTION_PATH = PARENT_ROOT / 'direction_library.json'
PARENT_ESTIMANDS_PATH = PARENT_ROOT / 'analysis' / 'mapping_averaged_estimands.csv'
PARENT_DECISION_PATH = PARENT_ROOT / 'analysis' / 'handoff_mediation_decision.json'

AUDIT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_error_conditioned_physical_state_audit_v1'
)
AUDIT_OUTCOME_MANIFEST_PATH = AUDIT_ROOT / 'outcome_prompt_manifest.jsonl'
AUDIT_OUTCOME_SHARD_DIR = AUDIT_ROOT / 'outcome_capture_shards'

DEPTH_ROOT = PROJECT_ROOT / 'results' / 'semantic_label_depth_localization_v1'
DEPTH_SUPPORT_PATH = DEPTH_ROOT / 'depth_support.jsonl'
INVERSE_SCENE_DIR = (
    PROJECT_ROOT / 'results' / 'inverse_query_weight_route_specificity_v1'
    / 'rendered_scenes'
)

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_mediator_matched_handoff_symmetry_v1'
)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
PAIR_MANIFEST_PATH = OUTPUT_ROOT / 'paired_original_mirror_manifest.jsonl'
MIRROR_DIR = OUTPUT_ROOT / 'mirrored_images'
BASELINE_PATH = OUTPUT_ROOT / 'baseline_mapping_replay.jsonl'
DIRECTION_LIBRARY_PATH = OUTPUT_ROOT / 'direction_library.json'
SHARD_DIR = OUTPUT_ROOT / 'paired_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for path in (OUTPUT_ROOT, MIRROR_DIR, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value, seed=SEED):
    return hashlib.sha256(f'{seed}:{value!r}'.encode('utf-8')).hexdigest()

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value) and value.numel() == 1:
        return value.item()
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')

def write_json_atomic(path, value):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    temporary.write_text(json.dumps(
        value, indent=2, ensure_ascii=False, default=json_default
    ) + '\n', encoding='utf-8')
    temporary.replace(path)

def write_jsonl_atomic(path, rows):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(path)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

required_inputs = (
    PARENT_RUN_PATH, PARENT_PROTOCOL_PATH, PARENT_SUPPORT_PATH,
    PARENT_SPLIT_PATH, PARENT_ROUTE_PATH, PARENT_DIRECTION_PATH,
    PARENT_ESTIMANDS_PATH, PARENT_DECISION_PATH,
    AUDIT_OUTCOME_MANIFEST_PATH, AUDIT_OUTCOME_SHARD_DIR,
    DEPTH_SUPPORT_PATH, INVERSE_SCENE_DIR,
)
for required in required_inputs:
    assert Path(required).exists(), f'Missing prerequisite: {required}'

parent_route = json.loads(PARENT_ROUTE_PATH.read_text(encoding='utf-8'))
assert parent_route['selected_layer'] == UPSTREAM_LAYER
assert parent_route['selected_method'] == UPSTREAM_METHOD
parent_direction = json.loads(PARENT_DIRECTION_PATH.read_text(encoding='utf-8'))
parent_estimands = pd.read_csv(PARENT_ESTIMANDS_PATH)
upstream = parent_estimands[
    parent_estimands['condition'].eq('upstream_exact')
].groupby('alpha', as_index=False)['l49_toward_exact'].mean().sort_values('alpha')
alpha_values = upstream['alpha'].to_numpy(dtype=float)
projection_values = upstream['l49_toward_exact'].to_numpy(dtype=float)
UPSTREAM_TO_L49_SLOPE = float(
    np.dot(alpha_values, projection_values) / np.dot(alpha_values, alpha_values)
)
PARENT_SEMANTIC_NORM = float(parent_direction['semantic_norm'])
MATCHED_UPSTREAM_ALPHA = float(PARENT_SEMANTIC_NORM / UPSTREAM_TO_L49_SLOPE)
assert 2.0 < MATCHED_UPSTREAM_ALPHA < 8.0, MATCHED_UPSTREAM_ALPHA

parent_primary = parent_estimands[np.isclose(parent_estimands['alpha'], 1.0)]
parent_l49 = parent_primary[parent_primary['condition'].eq('l49_exact')]
ANSWER_EQUIVALENCE_MARGIN = float(
    EQUIVALENCE_FRACTION * abs(parent_l49['margin_shift'].mean())
)
CODE_EQUIVALENCE_MARGIN = float(
    EQUIVALENCE_FRACTION * abs(parent_l49['l55_toward_exact_code'].mean())
)

CONDITION_SCALES = {
    'upstream_natural': 1.0,
    'upstream_halfmatched': 0.5 * MATCHED_UPSTREAM_ALPHA,
    'upstream_matched': MATCHED_UPSTREAM_ALPHA,
    'upstream_onehalfmatched': 1.5 * MATCHED_UPSTREAM_ALPHA,
    'upstream_reverse_matched': MATCHED_UPSTREAM_ALPHA,
    'upstream_random_matched': MATCHED_UPSTREAM_ALPHA,
    'l49_direct': 1.0, 'l49_reverse': 1.0, 'l49_random': 1.0,
    'l55_direct': 1.0, 'l55_stale': 1.0, 'l55_random': 1.0,
}

protocol_identity = {
    'schema_version': 'mediator-matched-handoff-symmetry-1.0',
    'analysis_status': 'fixed-route paired mirror confirmation',
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'parent_run_sha256': sha256_file(PARENT_RUN_PATH),
    'parent_route_sha256': sha256_file(PARENT_ROUTE_PATH),
    'route': {'layer': UPSTREAM_LAYER, 'method': UPSTREAM_METHOD},
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'gain_calibration': {
        'source': 'parent confirmation mapping-averaged alpha dose response',
        'upstream_to_l49_slope': UPSTREAM_TO_L49_SLOPE,
        'target_l49_projection': PARENT_SEMANTIC_NORM,
    },
    'conditions': CONDITION_SCALES,
    'mirror_selection_uses_behavior': False,
    'equivalence_fraction': EQUIVALENCE_FRACTION,
    'answer_equivalence_margin': ANSWER_EQUIVALENCE_MARGIN,
    'code_equivalence_margin': CODE_EQUIVALENCE_MARGIN,
    'bootstrap_unit': 'original-mirror physical pair',
    'seed': SEED,
}
canonical_identity = json.loads(json.dumps(protocol_identity, sort_keys=True))
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == canonical_identity, (
        'Existing protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': canonical_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Fixed route:', UPSTREAM_LAYER, UPSTREAM_METHOD)
print('Matched upstream alpha:', MATCHED_UPSTREAM_ALPHA)
print('Answer / L55 equivalence margins:',
      ANSWER_EQUIVALENCE_MARGIN, CODE_EQUIVALENCE_MARGIN)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 6/17 Build fixed original-mirror physical pairs
cell_stage('Mirroring every parent confirmation image without behavior filtering')
from collections import Counter, defaultdict
from dataclasses import replace as dataclass_replace
import shutil
from PIL import Image, ImageOps

from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')
scene_by_id = {str(row['scene_id']): row for row in scene_records}
loader_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_ROOT / '_unused_loader',
    model_id=MODEL_ID, model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,), question_kinds=QUESTION_KINDS,
    splits=('probe_train', 'probe_dev', 'probe_test'), batch_size=1,
    generation_every=0, seed=SEED,
)
all_examples = tuple(load_behavior_examples(loader_config))
examples_by_scene_kind = defaultdict(list)
for example in all_examples:
    examples_by_scene_kind[(str(example.scene_id), example.question_kind)].append(
        example
    )
prior_outcome_manifest = read_jsonl(AUDIT_OUTCOME_MANIFEST_PATH)
outcome_example_by_scene = {}
for row in prior_outcome_manifest:
    scene_id = str(row['scene_id'])
    candidates = examples_by_scene_kind[(scene_id, 'outcome')]
    outcome_example_by_scene[scene_id] = next(
        example for example in candidates if str(example.variant) == str(row['variant'])
    )

parent_support = read_jsonl(PARENT_SUPPORT_PATH)
parent_by_id = {row['scene_id']: row for row in parent_support}
parent_split = json.loads(PARENT_SPLIT_PATH.read_text(encoding='utf-8'))
confirmation_ids = list(parent_split['confirmation_scene_ids'])
assert len(confirmation_ids) == 16
original_manifest = [parent_by_id[scene_id] for scene_id in confirmation_ids]
assert all(row['exact_semantic'] == 'right' for row in original_manifest)

def swapped_side(value):
    return {'left': 'right', 'right': 'left', 'balanced': 'balanced'}[str(value)]

pair_manifest = []
scene_manifest = []
for original in tqdm(original_manifest, desc='Rendering mirror pairs', unit='pair'):
    scene_id = original['scene_id']
    pair_id = f'pair::{scene_id}'
    original_row = {
        **original, 'pair_id': pair_id, 'orientation': 'original',
        'source_scene_id': scene_id,
    }
    mirror_id = f'{scene_id}__horizontal_mirror'
    mirror_path = MIRROR_DIR / f'{mirror_id}.png'
    original_example = outcome_example_by_scene[scene_id]
    if not mirror_path.is_file():
        with Image.open(original_example.image_path) as image:
            ImageOps.mirror(image.convert('RGB')).save(mirror_path)
    mirror_row = {
        **original,
        'pair_id': pair_id, 'orientation': 'mirror',
        'source_scene_id': scene_id, 'scene_id': mirror_id,
        'exact_semantic': swapped_side(original['exact_semantic']),
        'shortcut_semantic': swapped_side(original['shortcut_semantic']),
        'outcome': swapped_side(original['outcome']),
        'net_torque': -int(original['net_torque']),
        'left_units': int(original['right_units']),
        'right_units': int(original['left_units']),
        'left_distance': int(original['right_distance']),
        'right_distance': int(original['left_distance']),
        'left_torque': int(original['right_torque']),
        'right_torque': int(original['left_torque']),
    }
    outcome_example_by_scene[mirror_id] = dataclass_replace(
        original_example, scene_id=mirror_id, image_path=mirror_path.resolve(),
        outcome=mirror_row['exact_semantic'],
        net_torque=int(mirror_row['net_torque']),
    )
    pair_manifest.append({
        'pair_id': pair_id, 'original_scene_id': scene_id,
        'mirror_scene_id': mirror_id,
    })
    scene_manifest.extend([original_row, mirror_row])

assert len(pair_manifest) == 16 and len(scene_manifest) == 32
assert Counter(row['exact_semantic'] for row in scene_manifest) == {
    'left': 16, 'right': 16,
}
write_jsonl_atomic(PAIR_MANIFEST_PATH, scene_manifest)
candidate_manifest = scene_manifest

direction_support = read_jsonl(DEPTH_SUPPORT_PATH)
EXPECTED_DIRECTION_FAMILIES = len(direction_support)
DIRECTION_RUNTIME_DIR = Path('/content/visualtorque_semantic_direction_images_v1')
DIRECTION_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
direction_files = sorted({
    role['image_filename']
    for row in direction_support for role in row['roles'].values()
})
for filename in tqdm(direction_files, desc='Caching direction images', unit='image'):
    source = INVERSE_SCENE_DIR / filename
    destination = DIRECTION_RUNTIME_DIR / filename
    assert source.is_file(), source
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
print('Pairs / physical scenes:', len(pair_manifest), len(scene_manifest))
print('Correct-side balance:', Counter(row['exact_semantic'] for row in scene_manifest))


## 2. Reconstructed directions and unfiltered symmetry audit


In [ ]:
%%cell_progress 7/17 Load pinned Qwen-27B and decoder layers
cell_stage(f'Loading {MODEL_KEY} weights for mediator-matched symmetry')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


In [ ]:
%%cell_progress 8/17 Define counterbalanced original and mirror prompts
cell_stage('Separating physical sides from A/B/C response codes')
import math
from dataclasses import replace as dataclass_replace
from visual_torque_toolkit.behavior.data import BehaviorExample

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = {
    'outcome': 'Which way will the balance scale tip?',
    'weight_more': 'Which side has more total weight?',
    'weight_less': 'Which side has less total weight?',
}
SEMANTIC_TEXT = {'left': 'left side', 'balanced': 'equal', 'right': 'right side'}

def outcome(value):
    return 'left' if value > 0 else 'right' if value < 0 else 'balanced'

def task_semantic(features, task):
    if task == 'outcome':
        value = int(features['net_torque'])
    elif task == 'weight_more':
        value = int(features['weight_score'])
    elif task == 'weight_less':
        value = -int(features['weight_score'])
    else:
        raise KeyError(task)
    return outcome(value)

def mapping_by_label(semantics):
    assert set(semantics) == {'left', 'balanced', 'right'}
    return tuple({
        'position': index, 'label': LABELS[index], 'semantic': semantic,
    } for index, semantic in enumerate(semantics))

def direction_reference_mapping(row):
    semantics = ['left', 'balanced', 'right']
    offset = int(stable_key(
        f"options:{row['family_id']}", seed=DIRECTION_SEED
    )[:8], 16) % 3
    return mapping_by_label(semantics[offset:] + semantics[:offset])

def target_mappings(row):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    assert {exact, wrong} == {'left', 'right'}
    balanced_label = LABELS[int(stable_key(
        ('balanced-label', row['scene_id'])
    )[:8], 16) % 3]
    side_labels = [label for label in LABELS if label != balanced_label]
    result = {}
    for variant, exact_label, wrong_label in (
        ('forward', side_labels[0], side_labels[1]),
        ('reverse', side_labels[1], side_labels[0]),
    ):
        by_label = {
            exact_label: exact, wrong_label: wrong, balanced_label: 'balanced'
        }
        result[variant] = mapping_by_label(tuple(by_label[label] for label in LABELS))
    return result

def semantics_by_label(mapping):
    return tuple(next(
        item['semantic'] for item in mapping if item['label'] == label
    ) for label in LABELS)

def label_for(mapping, semantic):
    return next(item['label'] for item in mapping if item['semantic'] == semantic)

def custom_prompt(task, mapping):
    options = '\n'.join(
        f"{item['label']}. {SEMANTIC_TEXT[item['semantic']]}" for item in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT, 'Question: ' + QUESTION_TEXT[task], 'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option letter '
        'and nothing else. Valid replies are: A, B, C.',
    ])

def make_direction_example(row, role_record, task, mapping, name):
    features = role_record['features']
    semantics = semantics_by_label(mapping)
    gold = task_semantic(features, task)
    return BehaviorExample(
        example_id=f"fracture_direction:{row['family_id']}:{name}",
        question_id=f"fracture_direction_{row['family_id']}_{name}",
        scene_id=str(role_record['scene_id']), condition=PromptCondition.IMAGE_RULE,
        question_kind=task, variant='fracture_direction', split='direction_support',
        problem_type='semantic_direction', outcome=gold,
        net_torque=int(features['net_torque']), prompt=custom_prompt(task, mapping),
        image_path=(DIRECTION_RUNTIME_DIR / role_record['image_filename']).resolve(),
        labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
        option_semantics=semantics, gold_label=LABELS[semantics.index(gold)],
        gold_semantic=gold,
    )

def build_direction_examples(row):
    mapping = direction_reference_mapping(row)
    examples = {}
    for task in DONOR_TASKS:
        for role in ('base', 'aligned_source'):
            key = f'{task}__{role}'
            examples[key] = make_direction_example(
                row, row['roles'][role], task, mapping, key
            )
    return examples, mapping

def build_target_examples(row):
    mappings = target_mappings(row)
    base = outcome_example_by_scene[row['scene_id']]
    examples = {}
    for variant, mapping in mappings.items():
        semantics = semantics_by_label(mapping)
        examples[variant] = dataclass_replace(
            base,
            example_id=f"fracture_target:{row['scene_id']}:{variant}",
            question_id=f"fracture_target_{row['scene_id']}_{variant}",
            variant=f'fracture_{variant}', prompt=custom_prompt('outcome', mapping),
            labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
            option_semantics=semantics,
            gold_label=LABELS[semantics.index(row['exact_semantic'])],
            gold_semantic=row['exact_semantic'],
        )
    return examples, mappings

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples], add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def final_mask(prepared):
    assert len(prepared['examples']) == 1
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    mask = torch.zeros_like(attention)
    mask[int(torch.where(attention)[0][-1])] = True
    return mask

print('Prompt builders ready.')


In [ ]:
%%cell_progress 9/17 Define intervention and downstream-state scorers
cell_stage('Defining variable-layer final-token transport')

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(scores))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label,
            'gold_semantic': example.gold_semantic,
        })
    return records

def forward_capture(prepared, layers):
    hidden_by_layer = {}
    handles = []
    for layer in layers:
        def hook(_module, _inputs, output, layer=layer):
            hidden_by_layer[layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_records(prepared, output.logits), hidden_by_layer

def capture_examples(examples, layers):
    scores = {}
    hidden = {name: {} for name in examples}
    masks = {}
    for name, example in examples.items():
        prepared = prepare_examples([example])
        masks[name] = final_mask(prepared)
        current_scores, current_hidden = forward_capture(prepared, layers)
        scores[name] = current_scores[0]
        for layer, tensor in current_hidden.items():
            hidden[name][layer] = tensor[0]
        del prepared, current_hidden
    return scores, hidden, masks

def selected_norm(tensor):
    return float(torch.linalg.vector_norm(tensor.float()))

def matched_random(tensor, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    noise = torch.randn(tensor.shape, generator=generator, dtype=torch.float32)
    return (noise * (selected_norm(tensor) / selected_norm(noise))).to(tensor.dtype)

def forward_intervention_chunk(prepared, conditions):
    assert len(prepared['examples']) == len(conditions)
    layers = {int(row['layer_index']) for row in conditions}
    assert len(layers) == 1
    layer = next(iter(layers))
    payloads = torch.stack([row['payload_selected'] for row in conditions])
    masks = torch.stack([row['target_mask'] for row in conditions])
    modes = [row['intervention_mode'] for row in conditions]
    counts = []
    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payloads.to(device=hidden.device, dtype=hidden.dtype)
        mask = masks.to(device=hidden.device)
        modified = hidden.clone()
        counts.clear()
        for index, mode in enumerate(modes):
            current = mask[index]
            assert int(current.sum()) == int(payload[index].shape[0]) == 1
            if mode == 'add':
                modified[index, current] = hidden[index, current] + payload[index]
            elif mode == 'replace':
                modified[index, current] = payload[index]
            else:
                raise ValueError(mode)
            counts.append(int(current.sum()))
        return replace_output_hidden(output, modified)
    handle = DECODER_LAYERS[layer].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition, count in zip(records, conditions, counts):
        record.update({
            key: value for key, value in condition.items()
            if key not in ('payload_selected', 'target_mask')
        })
        record['patched_tokens'] = count
    return records

def score_conditions(example, conditions):
    records = []
    grouped = defaultdict(list)
    for condition in conditions:
        grouped[int(condition['layer_index'])].append(condition)
    for layer_conditions in grouped.values():
        for start in range(0, len(layer_conditions), PATCH_CONDITION_BATCH_SIZE):
            actual = layer_conditions[start:start + PATCH_CONDITION_BATCH_SIZE]
            padded = list(actual)
            while len(padded) < PATCH_CONDITION_BATCH_SIZE:
                padded.append(padded[-1])
            prepared = prepare_examples([example] * len(padded))
            records.extend(forward_intervention_chunk(
                prepared, padded
            )[:len(actual)])
            del prepared
    return records

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def label_score(record, label):
    labels = list(record['candidate_labels'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    index = labels.index(label)
    other = np.delete(values, index)
    return float(values[index] - np.logaddexp.reduce(other))

def entropy(record):
    probabilities = np.asarray(record['candidate_probabilities'], dtype=float)
    return float(-np.sum(probabilities * np.log(np.clip(probabilities, 1e-12, 1))))

def cosine_similarity(left, right):
    left = left.float().reshape(-1)
    right = right.float().reshape(-1)
    denominator = torch.linalg.vector_norm(left) * torch.linalg.vector_norm(right)
    assert float(denominator) > 0
    return float(torch.dot(left, right) / denominator)

def rescale_direction(tensor, target_norm):
    norm = selected_norm(tensor)
    assert norm > 0 and target_norm > 0
    return (tensor.float() * (float(target_norm) / norm)).to(tensor.dtype)

def unit_direction(tensor):
    return rescale_direction(tensor, 1.0).float()

def score_l49_and_capture_l55(example, payload_selected, clean_l55, metadata):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    def semantic_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    def label_capture_hook(_module, _inputs, output):
        captured['hidden'] = output_hidden_tensor(output).detach().cpu().clone()
    semantic_handle = DECODER_LAYERS[SEMANTIC_LAYER].register_forward_hook(
        semantic_hook
    )
    label_handle = DECODER_LAYERS[LABEL_LAYER].register_forward_hook(
        label_capture_hook
    )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        semantic_handle.remove()
        label_handle.remove()
    record = score_records(prepared, output.logits)[0]
    downstream = captured['hidden'][0][mask] - clean_l55
    record.update(metadata)
    record['patched_tokens'] = int(mask.sum())
    record['downstream_l55_norm'] = selected_norm(downstream)
    return record, downstream

print('Semantic steering and downstream capture scorers ready.')


def semantic_margin(record, exact_semantic, wrong_semantic):
    semantics = list(record['candidate_semantics'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(
        values[semantics.index(exact_semantic)]
        - values[semantics.index(wrong_semantic)]
    )

def vector_projection(delta, direction):
    unit = unit_direction(direction).reshape(-1)
    return float(torch.dot(delta.float().reshape(-1), unit))

def forward_add_and_capture(example, layer, payload_selected, capture_layers):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    handles = []
    def intervention_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        assert int(current.sum()) == int(payload.shape[0]) == 1
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    handles.append(DECODER_LAYERS[layer].register_forward_hook(intervention_hook))
    for capture_layer in capture_layers:
        assert capture_layer > layer, (layer, capture_layer)
        def capture_hook(_module, _inputs, output, capture_layer=capture_layer):
            captured[capture_layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()[0]
        handles.append(
            DECODER_LAYERS[capture_layer].register_forward_hook(capture_hook)
        )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    record = score_records(prepared, output.logits)[0]
    return record, {
        capture_layer: tensor[mask] for capture_layer, tensor in captured.items()
    }, int(mask.sum())

print('Downstream state-capture intervention helper ready.')


In [ ]:
%%cell_progress 10/17 Rebuild the fixed L48, L49, and L55 directions
cell_stage('Reconstructing the parent route without new scene outcomes')
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

left_units, left_raw_norms = [], []
code_units, code_raw_norms = defaultdict(list), []
for row in tqdm(direction_support, desc='L49/L55 directions', unit='family'):
    examples, mapping = build_direction_examples(row)
    scores, hidden, masks = capture_examples(
        examples, (SEMANTIC_LAYER, LABEL_LAYER)
    )
    for task in DONOR_TASKS:
        base_key, source_key = f'{task}__base', f'{task}__aligned_source'
        base_semantic = task_semantic(row['roles']['base']['features'], task)
        source_semantic = task_semantic(
            row['roles']['aligned_source']['features'], task
        )
        semantic_delta = (
            hidden[source_key][SEMANTIC_LAYER][masks[source_key]]
            - hidden[base_key][SEMANTIC_LAYER][masks[base_key]]
        )
        left_delta = semantic_delta if source_semantic == 'left' else -semantic_delta
        left_units.append(unit_direction(left_delta))
        left_raw_norms.append(selected_norm(semantic_delta))
        code_delta = (
            hidden[source_key][LABEL_LAYER][masks[source_key]]
            - hidden[base_key][LABEL_LAYER][masks[base_key]]
        )
        transition = (
            label_for(mapping, base_semantic), label_for(mapping, source_semantic)
        )
        code_units[transition].append(unit_direction(code_delta))
        code_raw_norms.append(selected_norm(code_delta))
    del hidden
    torch.cuda.empty_cache()

SEMANTIC_STEERING_NORM = float(np.median(left_raw_norms))
CODE_STEERING_NORM = float(np.median(code_raw_norms))
SEMANTIC_DIRECTION = rescale_direction(
    torch.stack(left_units).mean(dim=0), SEMANTIC_STEERING_NORM
)
CODE_DIRECTIONS = {
    transition: rescale_direction(
        torch.stack(values).mean(dim=0), CODE_STEERING_NORM
    ) for transition, values in code_units.items()
}

audit_manifest = pd.DataFrame(read_jsonl(AUDIT_OUTCOME_MANIFEST_PATH))
hidden_rows, captured_ids = [], []
for path in tqdm(
    sorted(AUDIT_OUTCOME_SHARD_DIR.glob('outcome_*.pt')),
    desc='Loading prior L48 states', unit='shard',
):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    hidden_rows.append(payload['hidden'][str(UPSTREAM_LAYER)].float())
    captured_ids.extend(str(value) for value in payload['scene_ids'])
assert audit_manifest['scene_id'].astype(str).tolist() == captured_ids
X = torch.cat(hidden_rows, dim=0).numpy()
y = audit_manifest['net_torque'].to_numpy(dtype=float)
fit = audit_manifest['split'].isin(['probe_train', 'probe_dev']).to_numpy()
scaler = StandardScaler().fit(X[fit])
X_fit, y_fit = scaler.transform(X[fit]), y[fit]
pls = PLSRegression(n_components=4, scale=False, max_iter=1000)
pls.fit(X_fit, y_fit.reshape(-1, 1))
scores = pls.transform(X_fit)
latent_delta = scores[y_fit > 0].mean(axis=0) - scores[y_fit < 0].mean(axis=0)
score_centered = scores - scores.mean(axis=0, keepdims=True)
X_centered = X_fit - X_fit.mean(axis=0, keepdims=True)
decoder = np.linalg.lstsq(score_centered, X_centered, rcond=1e-5)[0]
raw_delta = (latent_delta @ decoder) * scaler.scale_
UPSTREAM_DIRECTION = torch.from_numpy(
    (raw_delta / 2).astype('float32')
).reshape(1, -1)

parent_upstream = next(
    row for row in parent_direction['upstream']
    if int(row['layer']) == UPSTREAM_LAYER and row['method'] == UPSTREAM_METHOD
)
assert abs(selected_norm(UPSTREAM_DIRECTION) - float(parent_upstream['norm'])) < 1e-4
assert abs(SEMANTIC_STEERING_NORM - PARENT_SEMANTIC_NORM) < 1e-4
direction_metadata = {
    'schema_version': 'fixed-mediator-matched-directions-1.0',
    'upstream_layer': UPSTREAM_LAYER, 'upstream_method': UPSTREAM_METHOD,
    'upstream_norm': selected_norm(UPSTREAM_DIRECTION),
    'semantic_norm': SEMANTIC_STEERING_NORM,
    'code_norm': CODE_STEERING_NORM,
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'parent_direction_sha256': sha256_file(PARENT_DIRECTION_PATH),
}
write_json_atomic(DIRECTION_LIBRARY_PATH, direction_metadata)
print(direction_metadata)


In [ ]:
%%cell_progress 11/17 Audit unfiltered original and mirror behavior
cell_stage('Scoring both answer mappings without selecting mirror outcomes')

baseline_rows = []
for row in tqdm(scene_manifest, desc='Original/mirror baselines', unit='scene'):
    examples, mappings = build_target_examples(row)
    scores, _, _ = capture_examples(examples, ())
    for variant in MAPPING_VARIANTS:
        score = scores[variant]
        baseline_rows.append({
            'pair_id': row['pair_id'], 'scene_id': row['scene_id'],
            'orientation': row['orientation'], 'mapping_variant': variant,
            'exact_semantic': row['exact_semantic'],
            'shortcut_semantic': row['shortcut_semantic'],
            'predicted_semantic': score['predicted_semantic'],
            'correct': bool(score['predicted_semantic'] == row['exact_semantic']),
            'predicted_shortcut': bool(
                score['predicted_semantic'] == row['shortcut_semantic']
            ),
        })
write_jsonl_atomic(BASELINE_PATH, baseline_rows)
baseline_frame = pd.DataFrame(baseline_rows)
display(baseline_frame.groupby('orientation').agg(
    mapping_runs=('correct', 'size'), accuracy=('correct', 'mean'),
    shortcut_rate=('predicted_shortcut', 'mean'),
).round(4))
print('No original or mirror scene is removed by this behavior audit.')


In [ ]:
%%cell_progress 12/17 Run parity, matching, mapping, and norm smoke tests
cell_stage('Checking fixed route reconstruction and matched intervention geometry')

smoke_row = scene_manifest[0]
smoke_examples, smoke_mappings = build_target_examples(smoke_row)
smoke_scores, smoke_hidden, smoke_masks = capture_examples(
    smoke_examples, CAPTURE_LAYERS
)
self_errors = {}
for layer in CAPTURE_LAYERS:
    for variant in MAPPING_VARIANTS:
        record = score_conditions(smoke_examples[variant], [{
            'condition': 'self_replay', 'layer_index': layer,
            'intervention_mode': 'replace',
            'payload_selected': smoke_hidden[variant][layer][smoke_masks[variant]],
            'target_mask': smoke_masks[variant],
        }])[0]
        error = float(np.max(np.abs(
            centered_scores(record) - centered_scores(smoke_scores[variant])
        )))
        assert error < SMOKE_CENTERED_LOGPROB_TOLERANCE
        self_errors[f'{layer}:{variant}'] = error

for name, tensor in (
    ('upstream', UPSTREAM_DIRECTION * MATCHED_UPSTREAM_ALPHA),
    ('semantic', SEMANTIC_DIRECTION),
    ('code', next(iter(CODE_DIRECTIONS.values()))),
):
    random = matched_random(tensor, ('smoke', name))
    relative = abs(selected_norm(random) - selected_norm(tensor)) / selected_norm(tensor)
    assert relative < NORM_RELATIVE_TOLERANCE, (name, relative)
assert Counter(row['exact_semantic'] for row in scene_manifest) == {
    'left': 16, 'right': 16,
}
print('Matched upstream alpha:', MATCHED_UPSTREAM_ALPHA)
print('Target L49 projection:', SEMANTIC_STEERING_NORM)
print('Maximum self-replay error:', max(self_errors.values()))


## 3. Mediator-matched paired confirmation


In [ ]:
%%cell_progress 13/17 Run fixed-dose paired mediator interventions
cell_stage('Processing original and mirror scenes with resumable shards')

capture_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'pair_manifest_sha256': sha256_file(PAIR_MANIFEST_PATH),
    'direction_sha256': sha256_file(DIRECTION_LIBRARY_PATH),
    'conditions': CONDITION_SCALES,
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()

for row in tqdm(scene_manifest, desc='Mediator-matched pairs', unit='scene'):
    shard_path = SHARD_DIR / f"{row['scene_id']}.json"
    if shard_path.is_file():
        existing = json.loads(shard_path.read_text(encoding='utf-8'))
        assert existing['capture_identity_sha256'] == CAPTURE_SHA256
        continue
    examples, mappings = build_target_examples(row)
    records = []
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    toward_exact_sign = 1.0 if exact == 'left' else -1.0
    for variant in MAPPING_VARIANTS:
        example, mapping = examples[variant], mappings[variant]
        clean_scores, clean_hidden, clean_masks = capture_examples(
            {variant: example}, CAPTURE_LAYERS
        )
        clean = clean_scores[variant]
        clean_margin = semantic_margin(clean, exact, wrong)
        clean_l49 = clean_hidden[variant][SEMANTIC_LAYER][clean_masks[variant]]
        clean_l55 = clean_hidden[variant][LABEL_LAYER][clean_masks[variant]]
        exact_label = label_for(mapping, exact)
        wrong_label = label_for(mapping, wrong)
        code_direction = CODE_DIRECTIONS[(wrong_label, exact_label)]
        other_variant = 'reverse' if variant == 'forward' else 'forward'
        other_mapping = mappings[other_variant]
        stale_transition = (
            label_for(other_mapping, wrong), label_for(other_mapping, exact)
        )

        upstream_exact = toward_exact_sign * UPSTREAM_DIRECTION
        l49_exact = toward_exact_sign * SEMANTIC_DIRECTION
        payloads = {
            'upstream_natural': upstream_exact,
            'upstream_halfmatched': upstream_exact * CONDITION_SCALES['upstream_halfmatched'],
            'upstream_matched': upstream_exact * CONDITION_SCALES['upstream_matched'],
            'upstream_onehalfmatched': upstream_exact * CONDITION_SCALES['upstream_onehalfmatched'],
            'upstream_reverse_matched': -upstream_exact * CONDITION_SCALES['upstream_reverse_matched'],
            'upstream_random_matched': matched_random(
                upstream_exact * CONDITION_SCALES['upstream_random_matched'],
                ('paired', row['scene_id'], variant, 'upstream'),
            ),
            'l49_direct': l49_exact,
            'l49_reverse': -l49_exact,
            'l49_random': matched_random(
                l49_exact, ('paired', row['scene_id'], variant, 'l49'),
            ),
            'l55_direct': code_direction,
            'l55_stale': CODE_DIRECTIONS[stale_transition],
            'l55_random': matched_random(
                code_direction, ('paired', row['scene_id'], variant, 'l55'),
            ),
        }
        for condition, payload in payloads.items():
            if condition.startswith('upstream_'):
                layer, capture_layers = UPSTREAM_LAYER, (SEMANTIC_LAYER, LABEL_LAYER)
            elif condition.startswith('l49_'):
                layer, capture_layers = SEMANTIC_LAYER, (LABEL_LAYER,)
            else:
                layer, capture_layers = LABEL_LAYER, ()
            record, downstream, patched = forward_add_and_capture(
                example, layer, payload, capture_layers
            )
            margin = semantic_margin(record, exact, wrong)
            if SEMANTIC_LAYER in downstream:
                l49_shift = toward_exact_sign * vector_projection(
                    downstream[SEMANTIC_LAYER] - clean_l49, SEMANTIC_DIRECTION
                )
            elif condition.startswith('l49_'):
                l49_shift = toward_exact_sign * vector_projection(
                    payload, SEMANTIC_DIRECTION
                )
            else:
                l49_shift = np.nan
            if LABEL_LAYER in downstream:
                l55_shift = vector_projection(
                    downstream[LABEL_LAYER] - clean_l55, code_direction
                )
            else:
                l55_shift = np.nan
            record.update({
                'pair_id': row['pair_id'], 'scene_id': row['scene_id'],
                'source_scene_id': row['source_scene_id'],
                'orientation': row['orientation'],
                'mapping_variant': variant,
                'diagnostic_cell': row['diagnostic_cell'],
                'condition': condition, 'layer_index': layer,
                'exact_semantic': exact, 'shortcut_semantic': wrong,
                'exact_label': exact_label, 'shortcut_label': wrong_label,
                'clean_margin': clean_margin,
                'margin_shift': margin - clean_margin,
                'l49_toward_exact': l49_shift,
                'l55_toward_exact_code': l55_shift,
                'rescued': bool(record['predicted_semantic'] == exact),
                'payload_norm': selected_norm(payload),
                'patched_tokens': patched,
            })
            records.append(record)
    write_json_atomic(shard_path, {
        'schema_version': 'mediator-matched-paired-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'scene_id': row['scene_id'], 'records': records,
    })
    torch.cuda.empty_cache()
print('Completed scene shards:', len(list(SHARD_DIR.glob('*.json'))))


In [ ]:
%%cell_progress 14/17 Assemble mapping-averaged paired estimands
cell_stage('Reducing answer mappings while preserving original-mirror pairs')

all_records = []
for row in scene_manifest:
    payload = json.loads(
        (SHARD_DIR / f"{row['scene_id']}.json").read_text(encoding='utf-8')
    )
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    all_records.extend(payload['records'])
record_frame = pd.DataFrame(all_records)
record_frame.to_csv(ANALYSIS_DIR / 'paired_intervention_records.csv', index=False)
estimand_frame = record_frame.groupby(
    ['pair_id', 'scene_id', 'source_scene_id', 'orientation',
     'diagnostic_cell', 'condition'], as_index=False
).agg(
    margin_shift=('margin_shift', 'mean'),
    l49_toward_exact=('l49_toward_exact', 'mean'),
    l55_toward_exact_code=('l55_toward_exact_code', 'mean'),
    rescued=('rescued', 'mean'), payload_norm=('payload_norm', 'mean'),
)
estimand_frame.to_csv(ANALYSIS_DIR / 'mapping_averaged_estimands.csv', index=False)
assert estimand_frame['pair_id'].nunique() == 16
display(estimand_frame.groupby(['orientation', 'condition'])[
    ['margin_shift', 'l49_toward_exact', 'l55_toward_exact_code', 'rescued']
].mean().round(4))


In [ ]:
%%cell_progress 15/17 Test symmetry and mediator equivalence
cell_stage('Bootstrapping original-mirror physical pairs')
import gc

rng = np.random.default_rng(SEED)

def bootstrap_summary(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    assert len(values) == 16, len(values)
    draws = rng.choice(
        values, size=(BOOTSTRAP_REPLICATES, len(values)), replace=True
    ).mean(axis=1)
    return {
        'estimate': float(values.mean()),
        'ci_low': float(np.quantile(draws, ALPHA_LEVEL / 2)),
        'ci_high': float(np.quantile(draws, 1 - ALPHA_LEVEL / 2)),
        'n_pairs': int(len(values)),
    }

balanced_frame = estimand_frame.groupby(
    ['pair_id', 'condition'], as_index=False
)[['margin_shift', 'l49_toward_exact', 'l55_toward_exact_code']].mean()
mirror_frame = estimand_frame[estimand_frame['orientation'].eq('mirror')].copy()
original_frame = estimand_frame[estimand_frame['orientation'].eq('original')].copy()

def pivot(frame, metric):
    return frame.pivot(index='pair_id', columns='condition', values=metric)

tests = []
mirror_l49 = pivot(mirror_frame, 'l49_toward_exact')
for name, values in (
    ('mirror_matched_shift', mirror_l49['upstream_matched']),
    ('mirror_matched_minus_random',
     mirror_l49['upstream_matched'] - mirror_l49['upstream_random_matched']),
    ('mirror_matched_minus_reverse',
     mirror_l49['upstream_matched'] - mirror_l49['upstream_reverse_matched']),
):
    tests.append({'family': 'mirror_transport', 'metric': name,
                  **bootstrap_summary(values)})

equivalence_specs = (
    ('l49_coordinate', 'l49_toward_exact',
     MEDIATOR_MATCH_FRACTION * SEMANTIC_STEERING_NORM),
    ('l55_code', 'l55_toward_exact_code', CODE_EQUIVALENCE_MARGIN),
    ('answer_margin', 'margin_shift', ANSWER_EQUIVALENCE_MARGIN),
)
equivalence = {}
for family, metric, margin in equivalence_specs:
    current = pivot(balanced_frame, metric)
    difference = current['upstream_matched'] - current['l49_direct']
    result = bootstrap_summary(difference)
    passed = bool(result['ci_low'] > -margin and result['ci_high'] < margin)
    equivalence[family] = {'margin': float(margin), 'passed': passed, **result}
    tests.append({
        'family': 'mediator_equivalence', 'metric': family,
        'equivalence_margin': float(margin),
        'equivalence_passed': passed, **result,
    })

# Side symmetry is reported as an original-minus-mirror paired difference.
for metric in ('l49_toward_exact', 'l55_toward_exact_code', 'margin_shift'):
    original = pivot(original_frame, metric)['upstream_matched']
    mirror = pivot(mirror_frame, metric)['upstream_matched']
    tests.append({
        'family': 'side_symmetry', 'metric': metric + '_original_minus_mirror',
        **bootstrap_summary(original - mirror),
    })

test_frame = pd.DataFrame(tests)
test_frame.to_csv(ANALYSIS_DIR / 'paired_bootstrap_tests.csv', index=False)
lookup = test_frame.set_index(['family', 'metric'])

def lower(family, metric):
    return float(lookup.loc[(family, metric), 'ci_low'])

mirror_transport_gate = bool(
    lower('mirror_transport', 'mirror_matched_shift') > 0
    and lower('mirror_transport', 'mirror_matched_minus_random') > 0
    and lower('mirror_transport', 'mirror_matched_minus_reverse') > 0
)
mediator_match_gate = equivalence['l49_coordinate']['passed']
downstream_equivalence_gate = bool(
    equivalence['l55_code']['passed']
    and equivalence['answer_margin']['passed']
)

# Direct L49 must remain causally effective on mirrors.
mirror_l55 = pivot(mirror_frame, 'l55_toward_exact_code')
mirror_answer = pivot(mirror_frame, 'margin_shift')
direct_l49_l55 = bootstrap_summary(mirror_l55['l49_direct'])
direct_l49_answer = bootstrap_summary(mirror_answer['l49_direct'])
direct_l49_gate = bool(
    direct_l49_l55['ci_low'] > 0 and direct_l49_answer['ci_low'] > 0
)

if mirror_transport_gate and mediator_match_gate and downstream_equivalence_gate:
    PRIMARY_DECISION = 'l49_coordinate_is_sufficient_symmetric_mediator'
elif mirror_transport_gate and mediator_match_gate and direct_l49_gate:
    PRIMARY_DECISION = 'symmetric_transport_but_one_dimensional_l49_mediator_incomplete'
elif not mirror_transport_gate:
    PRIMARY_DECISION = 'parent_route_does_not_replicate_on_left_mirrors'
else:
    PRIMARY_DECISION = 'global_gain_failed_to_match_l49_mediator'

decision = {
    'primary_decision': PRIMARY_DECISION,
    'gates': {
        'mirror_transport': mirror_transport_gate,
        'l49_mediator_matched': mediator_match_gate,
        'downstream_equivalence': downstream_equivalence_gate,
        'direct_l49_effective_on_mirrors': direct_l49_gate,
    },
    'equivalence': equivalence,
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'mirror_baseline_accuracy': float(
        baseline_frame[baseline_frame['orientation'].eq('mirror')]['correct'].mean()
    ),
    'claim_boundary': (
        'Mirrors balance physical direction but are transformed versions of '
        'the same geometries. Equivalence concerns the measured final-token '
        'L49 direction, not the complete L49 hidden state.'
    ),
}
write_json_atomic(ANALYSIS_DIR / 'mediator_symmetry_decision.json', decision)
display(test_frame.round(4))
print('Gates:', decision['gates'])
print('Primary decision:', PRIMARY_DECISION)

if 'adapter' in globals():
    del adapter
gc.collect()
torch.cuda.empty_cache()


In [ ]:
%%cell_progress 16/17 Audit pairs, no filtering, mappings, doses, and replay
cell_stage('Checking the paired causal claim boundary')

assert len(pair_manifest) == 16
assert len(scene_manifest) == 32
assert Counter(row['orientation'] for row in scene_manifest) == {
    'original': 16, 'mirror': 16,
}
assert Counter(row['exact_semantic'] for row in scene_manifest) == {
    'left': 16, 'right': 16,
}
assert len(list(MIRROR_DIR.glob('*.png'))) >= 16
assert len(baseline_frame) == len(scene_manifest) * len(MAPPING_VARIANTS)
assert record_frame['scene_id'].nunique() == 32
assert set(record_frame['mapping_variant']) == set(MAPPING_VARIANTS)
assert record_frame['patched_tokens'].eq(1).all()
assert set(record_frame['condition']) == set(CONDITION_SCALES)
assert record_frame.groupby(
    ['scene_id', 'mapping_variant', 'condition']
).size().eq(1).all()

norm_groups = (
    ('upstream_matched', 'upstream_reverse_matched', 'upstream_random_matched'),
    ('l49_direct', 'l49_reverse', 'l49_random'),
    ('l55_direct', 'l55_stale', 'l55_random'),
)
norm_rows = []
for keys, current in record_frame.groupby(['scene_id', 'mapping_variant']):
    values_by_condition = current.set_index('condition')['payload_norm']
    for index, conditions in enumerate(norm_groups):
        values = values_by_condition.loc[list(conditions)].to_numpy(dtype=float)
        relative_range = float((values.max() - values.min()) / values.mean())
        norm_rows.append({
            'scene_id': keys[0], 'mapping_variant': keys[1],
            'control_family': index, 'relative_range': relative_range,
        })
norm_frame = pd.DataFrame(norm_rows)
assert float(norm_frame['relative_range'].max()) < NORM_RELATIVE_TOLERANCE
norm_frame.to_csv(ANALYSIS_DIR / 'norm_audit.csv', index=False)

audit_summary = {
    'physical_pairs': 16, 'physical_scenes': 32,
    'correct_left': 16, 'correct_right': 16,
    'mirror_behavior_used_for_selection': False,
    'answer_mapping_variants': 2,
    'fixed_parent_route': [UPSTREAM_LAYER, UPSTREAM_METHOD],
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'maximum_self_replay_centered_error': max(self_errors.values()),
    'maximum_control_norm_relative_range': float(
        norm_frame['relative_range'].max()
    ),
}
write_json_atomic(ANALYSIS_DIR / 'mediator_symmetry_audit.json', audit_summary)
print(audit_summary)


In [ ]:
%%cell_progress 17/17 Save the mediator-matching figure and report
cell_stage('Writing paper-facing paired symmetry evidence')
import matplotlib.pyplot as plt

dose_conditions = [
    'upstream_natural', 'upstream_halfmatched',
    'upstream_matched', 'upstream_onehalfmatched',
]
dose_labels = ['1× natural', '0.5× matched', 'matched', '1.5× matched']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.7))
for orientation, color in (('original', '#2563EB'), ('mirror', '#D97706')):
    current = estimand_frame[estimand_frame['orientation'].eq(orientation)]
    means = current.groupby('condition')['l49_toward_exact'].mean()
    axes[0].plot(
        range(len(dose_conditions)),
        [means[condition] for condition in dose_conditions],
        marker='o', linewidth=2, color=color, label=orientation.title(),
    )
axes[0].axhline(SEMANTIC_STEERING_NORM, color='black', linestyle='--',
                label='Direct L49 target')
axes[0].set_xticks(range(len(dose_conditions)), dose_labels, rotation=20)
axes[0].set_ylabel('L49 movement toward correct side')
axes[0].set_title('A. Globally calibrated mediator dose')
axes[0].legend(frameon=False)

comparison = ['upstream_matched', 'l49_direct']
labels = ['L48 matched', 'L49 direct']
balanced_means = balanced_frame.groupby('condition').mean(numeric_only=True)
axes[1].bar(
    labels,
    [balanced_means.loc[c, 'l55_toward_exact_code'] for c in comparison],
    color=['#0F766E', '#7C3AED'],
)
axes[1].set_ylabel('L55 movement toward correct current code')
axes[1].set_title('B. Downstream state at matched L49')
axes[2].bar(
    labels,
    [balanced_means.loc[c, 'margin_shift'] for c in comparison],
    color=['#0F766E', '#7C3AED'],
)
axes[2].set_ylabel('Correct-vs-wrong answer log-odds shift')
axes[2].set_title('C. Behavioral mediation')
fig.suptitle(PRIMARY_DECISION, y=1.03)
fig.tight_layout()
figure_path = FIGURE_DIR / 'mediator_matched_handoff_symmetry.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()

def interval(family, metric):
    row = test_frame.set_index(['family', 'metric']).loc[(family, metric)]
    return f"{row['estimate']:.5f} [{row['ci_low']:.5f}, {row['ci_high']:.5f}]"

lines = [
    '# Mediator-Matched Handoff Symmetry', '',
    f'**Primary decision:** `{PRIMARY_DECISION}`', '',
    '## Design', '',
    '- Fixed parent route: L48 `pls4_prototype`',
    f'- Parent-calibrated matched alpha: {MATCHED_UPSTREAM_ALPHA:.5f}',
    '- Original/mirror physical pairs: 16',
    '- Correct-side balance: 16 left, 16 right',
    '- Mirror behavior was measured but never used for selection.', '',
    '## Mirror transport', '',
    f"- Matched L48 shift: {interval('mirror_transport', 'mirror_matched_shift')}",
    f"- Matched minus random: {interval('mirror_transport', 'mirror_matched_minus_random')}",
    f"- Matched minus reverse: {interval('mirror_transport', 'mirror_matched_minus_reverse')}",
    '', '## Mediator equivalence', '',
    f"- L49 coordinate difference: {interval('mediator_equivalence', 'l49_coordinate')}",
    f"- L55 code difference: {interval('mediator_equivalence', 'l55_code')}",
    f"- Answer-margin difference: {interval('mediator_equivalence', 'answer_margin')}",
    '', '## Claim boundary', '',
    (
        'The mirror test balances physical direction but reuses transformed '
        'geometries. Mediator matching concerns one frozen final-token L49 '
        'coordinate. Failure of downstream equivalence means that this '
        'coordinate does not summarize the complete mediating state.'
    ), '',
]
report_path = ANALYSIS_DIR / 'RESULT_CARD.md'
report_path.write_text('\n'.join(lines), encoding='utf-8')
run_complete = {
    'status': 'complete',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'primary_decision': PRIMARY_DECISION,
    'physical_pairs': 16, 'physical_scenes': 32,
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'decision_sha256': sha256_file(
        ANALYSIS_DIR / 'mediator_symmetry_decision.json'
    ),
    'figure': str(figure_path), 'report': str(report_path),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', run_complete)
print('Saved:', figure_path)
print('Saved:', report_path)
print('RUN_COMPLETE:', OUTPUT_ROOT / 'RUN_COMPLETE.json')
